# 12 - Confidence Granularity Ablation

**Project:** MARS - Multi-Agent Recommender System  
**Purpose:** quantify the impact of 2/3/4/6 confidence classes on confidence quality, prediction AUC-ROC, recommendation NDCG@10, and an offline learning-gain proxy.

In [ ]:
import sys
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from scipy.stats import spearmanr, wilcoxon
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score
from torch.utils.data import DataLoader

warnings.filterwarnings('ignore')

ROOT = Path.cwd().resolve()
PROJECT_ROOT = ROOT.parent if ROOT.name == 'notebooks' else ROOT
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from agents.confidence_agent import ConfidenceAgent, get_confidence_schema
from agents.kg_agent import KnowledgeGraphAgent
from agents.prediction_agent import GapSequenceDataset, PredictionAgent
from agents.recommendation_agent import Rec, RecommendationAgent
from agents.utils import set_global_seed

RESULTS_DIR = PROJECT_ROOT / 'results' / 'confidence_ablation'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SCHEMAS = [2, 3, 4, 6]
SEEDS = [42, 123, 456, 789, 2024]
TOP_K = 10
MAX_RANK_USERS = 120
MAX_EVAL_USERS = 200

plt.style.use('seaborn-v0_8-paper')
sns.set_theme(style='whitegrid')
print(f'Project root: {PROJECT_ROOT}')
print(f'Results dir:  {RESULTS_DIR}')

In [ ]:
interactions = pd.read_parquet(PROJECT_ROOT / 'data' / 'processed' / 'interactions.parquet')
questions = pd.read_csv(PROJECT_ROOT / 'data' / 'raw' / 'questions.csv')
lectures = pd.read_csv(PROJECT_ROOT / 'data' / 'raw' / 'lectures.csv')

interactions = interactions.sort_values(['user_id', 'timestamp']).reset_index(drop=True)
users = interactions['user_id'].drop_duplicates().to_numpy()
rng = np.random.default_rng(42)
users = rng.permutation(users)

n_users = len(users)
n_train = int(n_users * 0.70)
n_val = int(n_users * 0.15)
train_users = set(users[:n_train])
val_users = set(users[n_train:n_train + n_val])
test_users = set(users[n_train + n_val:])

train_df = interactions[interactions['user_id'].isin(train_users)].copy()
val_df = interactions[interactions['user_id'].isin(val_users)].copy()
test_df = interactions[interactions['user_id'].isin(test_users)].copy()

print(f'Interactions: {len(interactions):,}')
print(f'Users: {n_users:,}')
print(f'Train/val/test users: {len(train_users)}/{len(val_users)}/{len(test_users)}')
print(f'Train/val/test rows: {len(train_df):,}/{len(val_df):,}/{len(test_df):,}')

kg_agent = KnowledgeGraphAgent()
kg_agent.build_graph(questions, lectures)
kg_agent.update_difficulties(train_df)
kg_agent.build_prerequisites(train_df)
print('Shared KG artefacts built once on the fixed training split.')

In [ ]:
def parse_tags(value):
    if isinstance(value, list):
        return [int(v) for v in value]
    if isinstance(value, str):
        parts = value.replace(';', ',').split(',')
        return [int(p.strip()) for p in parts if p.strip().isdigit()]
    if pd.notna(value) and isinstance(value, (int, float)):
        return [int(value)]
    return []

def normalise_question_ids(values):
    out = set()
    for value in values:
        s = str(value)
        out.add(s)
        if s.startswith('q'):
            out.add(s[1:])
        else:
            out.add(f'q{s}')
    return out

def ndcg_at_k(rec_ids, ground_truth_ids, k=10):
    rec_ids = list(rec_ids)[:k]
    gains = [1.0 if rid in ground_truth_ids else 0.0 for rid in rec_ids]
    if not gains:
        return 0.0
    dcg = sum(g / np.log2(i + 2) for i, g in enumerate(gains))
    ideal_len = min(len(ground_truth_ids), k)
    if ideal_len == 0:
        return 0.0
    idcg = sum(1.0 / np.log2(i + 2) for i in range(ideal_len))
    return float(dcg / idcg)

def attach_confidence_predictions(conf_agent, df, batch_size=4096):
    df = df.copy().reset_index(drop=True)
    classes = []
    names = []
    for start in range(0, len(df), batch_size):
        batch = df.iloc[start:start + batch_size]
        result = conf_agent.classify_batch(interactions=batch)
        classes.extend(result['classes'])
        names.extend(result['class_names'])
    df['confidence_class'] = classes
    df['confidence_label'] = names
    return df

def evaluate_confusion(conf_agent, df):
    labels_true = conf_agent._assign_labels(df).to_numpy()
    pred = conf_agent.classify_batch(interactions=df)
    labels_pred = np.array(pred['classes'])
    cm = confusion_matrix(labels_true, labels_pred, labels=list(range(conf_agent.n_classes)))
    return cm, f1_score(labels_true, labels_pred, average='macro', zero_division=0)

def evaluate_prediction_agent(pred_agent, df):
    dataset = GapSequenceDataset(df)
    if len(dataset) == 0:
        return {'auc_roc': np.nan, 'n_sequences': 0}
    loader = DataLoader(dataset, batch_size=512, shuffle=False)
    y_true = []
    y_score = []
    pred_agent.model.eval()
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(next(pred_agent.model.parameters()).device)
            scores = torch.sigmoid(pred_agent.model(xb)).cpu().numpy()
            y_true.append(yb.numpy())
            y_score.append(scores)
    y_true = np.vstack(y_true)
    y_score = np.vstack(y_score)
    valid_mask = (y_true.sum(axis=0) > 0) & (y_true.sum(axis=0) < len(y_true))
    auc = np.nan
    if valid_mask.any():
        auc = roc_auc_score(y_true[:, valid_mask], y_score[:, valid_mask], average='macro')
    return {'auc_roc': float(auc), 'n_sequences': int(len(dataset))}

def compute_learning_gain_proxy(context_df, future_df, related_tags):
    if not related_tags:
        return np.nan
    related_tags = set(int(t) for t in related_tags)
    context_mask = context_df['tags'].apply(lambda x: bool(related_tags.intersection(parse_tags(x))))
    future_mask = future_df['tags'].apply(lambda x: bool(related_tags.intersection(parse_tags(x))))
    if not future_mask.any():
        return np.nan
    context_acc = context_df.loc[context_mask, 'correct'].mean() if context_mask.any() else context_df['correct'].mean()
    future_acc = future_df.loc[future_mask, 'correct'].mean()
    return float(future_acc - context_acc)

def build_ranker_training_data(rec_agent, conf_agent, train_aug, sample_users=MAX_RANK_USERS):
    feature_blocks = []
    label_blocks = []
    groups = []
    user_ids = train_aug['user_id'].drop_duplicates().astype(str).tolist()[:sample_users]
    for uid in user_ids:
        user_df = train_aug[train_aug['user_id'].astype(str) == uid].sort_values('timestamp')
        if len(user_df) < 10:
            continue
        split = max(5, int(len(user_df) * 0.5))
        context = user_df.iloc[:split].copy()
        future = user_df.iloc[split:].copy()
        if future.empty:
            continue
        conf_result = conf_agent.classify_batch(user_id=uid, interactions=context)
        rec_agent.update_user_profile(uid, interactions=context, confidence_result=conf_result)
        diag_responses = [
            {'question_id': row['question_id'], 'correct': bool(row['correct'])}
            for _, row in context.head(15).iterrows()
        ]
        kg_profile = kg_agent.handle_cold_start(uid, diagnostic={'responses': diag_responses}, confidence=conf_result)
        recs = rec_agent.recommend(uid, kg_profile=kg_profile, confidence=conf_result, n=20)
        items = recs.get('items', [])
        if not items:
            continue
        candidates = [
            Rec(
                item_id=str(item['item_id']),
                item_type=item['item_type'],
                score=float(item['score']),
                strategy=item['strategy'],
                related_tags=item.get('related_tags', []),
            )
            for item in items
        ]
        X = rec_agent._build_ranking_features(uid, candidates)
        gt_ids = normalise_question_ids(future['question_id'].tolist())
        y = np.array([1.0 if str(item['item_id']) in gt_ids else 0.0 for item in items], dtype=np.float32)
        feature_blocks.append(X)
        label_blocks.append(y)
        groups.append(len(items))
    if not feature_blocks:
        return None, None, None
    return np.vstack(feature_blocks), np.concatenate(label_blocks), groups

def evaluate_recommendation_agent(rec_agent, conf_agent, df, top_k=TOP_K, max_users=MAX_EVAL_USERS):
    ndcgs = []
    gains = []
    user_ids = df['user_id'].drop_duplicates().astype(str).tolist()[:max_users]
    for uid in user_ids:
        user_df = df[df['user_id'].astype(str) == uid].sort_values('timestamp')
        if len(user_df) < 10:
            continue
        split = max(5, int(len(user_df) * 0.6))
        context = user_df.iloc[:split].copy()
        future = user_df.iloc[split:].copy()
        if future.empty:
            continue
        conf_result = conf_agent.classify_batch(user_id=uid, interactions=context)
        rec_agent.update_user_profile(uid, interactions=context, confidence_result=conf_result)
        diag_responses = [
            {'question_id': row['question_id'], 'correct': bool(row['correct'])}
            for _, row in context.head(15).iterrows()
        ]
        kg_profile = kg_agent.handle_cold_start(uid, diagnostic={'responses': diag_responses}, confidence=conf_result)
        recs = rec_agent.recommend(uid, kg_profile=kg_profile, confidence=conf_result, n=top_k)
        items = recs.get('items', [])
        rec_ids = [str(item['item_id']) for item in items]
        gt_ids = normalise_question_ids(future['question_id'].tolist())
        ndcgs.append(ndcg_at_k(rec_ids, gt_ids, k=top_k))
        related_tags = sorted({int(tag) for item in items for tag in item.get('related_tags', [])})
        gains.append(compute_learning_gain_proxy(context, future, related_tags))
    return {
        'ndcg_at_10': float(np.mean(ndcgs)) if ndcgs else np.nan,
        'learning_gain': float(np.nanmean(gains)) if gains else np.nan,
        'n_eval_users': len(ndcgs),
    }

def validate_skill_deltas(df, n_classes=6):
    schema = get_confidence_schema(n_classes)
    class_means = []
    delta_values = []
    work = df.sort_values(['user_id', 'timestamp']).copy()
    for class_id, class_name in enumerate(schema['class_names']):
        deltas = []
        for _, user_df in work.groupby('user_id'):
            rows = user_df.index[user_df['confidence_class'] == class_id].tolist()
            for idx in rows:
                pos = user_df.index.get_loc(idx)
                tag = parse_tags(user_df.loc[idx, 'tags'])
                if not tag:
                    continue
                tag = tag[0]
                prev_window = user_df.iloc[max(0, pos - 10):pos]
                next_window = user_df.iloc[pos + 1:pos + 11]
                prev_tag = prev_window[prev_window['tags'].apply(lambda x: tag in parse_tags(x))]
                next_tag = next_window[next_window['tags'].apply(lambda x: tag in parse_tags(x))]
                if prev_tag.empty or next_tag.empty:
                    continue
                deltas.append(float(next_tag['correct'].mean() - prev_tag['correct'].mean()))
        if deltas:
            class_means.append(float(np.mean(deltas)))
            delta_values.append(schema['skill_deltas_by_id'][class_id])
    if len(class_means) < 3:
        return {'spearman_rho': np.nan, 'p_value': np.nan}
    rho, p_value = spearmanr(delta_values, class_means)
    return {'spearman_rho': float(rho), 'p_value': float(p_value)}

In [ ]:
def run_single_experiment(n_classes, seed):
    set_global_seed(seed)
    conf_agent = ConfidenceAgent(n_classes=n_classes)
    conf_agent._config['seed'] = seed
    conf_metrics = conf_agent.train(train_df)

    train_aug = attach_confidence_predictions(conf_agent, train_df)
    val_aug = attach_confidence_predictions(conf_agent, val_df)
    test_aug = attach_confidence_predictions(conf_agent, test_df)

    pred_agent = PredictionAgent(model_type='lstm')
    pred_agent._config['seed'] = seed
    pred_agent.set_confidence_schema(n_classes)
    pred_agent.train(train_aug, val_df=val_aug)
    pred_metrics = evaluate_prediction_agent(pred_agent, test_aug)

    rec_agent = RecommendationAgent(random_seed=seed)
    rec_agent.set_confidence_schema(n_classes)
    rec_agent.build_content_index(lectures, questions)
    rec_agent.train_collaborative(train_aug)
    rank_X, rank_y, rank_groups = build_ranker_training_data(rec_agent, conf_agent, train_aug)
    if rank_X is not None:
        rec_agent.train_ranker(rank_X, rank_y, rank_groups)
    rec_metrics = evaluate_recommendation_agent(rec_agent, conf_agent, test_aug)

    cm, conf_f1_eval = evaluate_confusion(conf_agent, test_df)
    label_dist = pd.Series(train_aug['confidence_label']).value_counts(normalize=True).to_dict()

    return {
        'n_classes': n_classes,
        'seed': seed,
        'conf_f1_macro': float(conf_metrics.get('cv_f1_macro_mean', np.nan)),
        'conf_f1_macro_eval': float(conf_f1_eval),
        'auc_roc': float(pred_metrics['auc_roc']),
        'ndcg_at_10': float(rec_metrics['ndcg_at_10']),
        'learning_gain': float(rec_metrics['learning_gain']),
        'n_eval_users': int(rec_metrics['n_eval_users']),
        'confusion_matrix': cm,
        'label_distribution': label_dist,
    }

raw_results = []
cm_store = {}
dist_store = {}

for n_classes in SCHEMAS:
    for seed in SEEDS:
        t0 = time.time()
        result = run_single_experiment(n_classes, seed)
        raw_results.append({k: v for k, v in result.items() if k not in {'confusion_matrix', 'label_distribution'}})
        cm_store[(n_classes, seed)] = result['confusion_matrix']
        dist_store[(n_classes, seed)] = result['label_distribution']
        print(
            f'n_classes={n_classes} seed={seed} | '
            f'ConfF1={result["conf_f1_macro"]:.4f} | '
            f'AUC={result["auc_roc"]:.4f} | '
            f'NDCG@10={result["ndcg_at_10"]:.4f} | '
            f'Gain={result["learning_gain"]:.4f} | '
            f'{time.time() - t0:.1f}s'
        )

results_df = pd.DataFrame(raw_results)
results_df.to_csv(RESULTS_DIR / 'confidence_ablation_raw.csv', index=False)
results_df.head()

In [ ]:
summary_rows = []
binary_ndcg = results_df[results_df['n_classes'] == 2]['ndcg_at_10']
binary_mean = float(binary_ndcg.mean()) if len(binary_ndcg) else np.nan

for n_classes in SCHEMAS:
    subset = results_df[results_df['n_classes'] == n_classes]
    ndcg_mean = float(subset['ndcg_at_10'].mean())
    summary_rows.append({
        'n_classes': n_classes,
        'Conf F1-macro': f"{subset['conf_f1_macro'].mean():.4f} +- {subset['conf_f1_macro'].std(ddof=1):.4f}",
        'NDCG@10': f"{ndcg_mean:.4f} +- {subset['ndcg_at_10'].std(ddof=1):.4f}",
        'AUC-ROC': f"{subset['auc_roc'].mean():.4f} +- {subset['auc_roc'].std(ddof=1):.4f}",
        'Learning Gain': f"{subset['learning_gain'].mean():.4f} +- {subset['learning_gain'].std(ddof=1):.4f}",
        'Delta vs binary (%)': np.nan if n_classes == 2 else round(100.0 * (ndcg_mean - binary_mean) / max(binary_mean, 1e-8), 2),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(RESULTS_DIR / 'confidence_ablation_summary.csv', index=False)
display(summary_df)

wilcoxon_rows = []
base = results_df[results_df['n_classes'] == 6].sort_values('seed')
for other in [2, 3, 4]:
    comp = results_df[results_df['n_classes'] == other].sort_values('seed')
    merged = base[['seed', 'ndcg_at_10']].merge(comp[['seed', 'ndcg_at_10']], on='seed', suffixes=('_6c', f'_{other}c'))
    if len(merged) >= 3:
        stat, p_value = wilcoxon(merged['ndcg_at_10_6c'], merged[f'ndcg_at_10_{other}c'])
    else:
        stat, p_value = np.nan, np.nan
    wilcoxon_rows.append({'comparison': f'6 vs {other}', 'statistic': stat, 'p_value': p_value})

wilcoxon_df = pd.DataFrame(wilcoxon_rows)
wilcoxon_df.to_csv(RESULTS_DIR / 'confidence_ablation_wilcoxon.csv', index=False)
display(wilcoxon_df)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

plot_df = results_df.groupby('n_classes', as_index=False).agg(
    ndcg_mean=('ndcg_at_10', 'mean'),
    ndcg_std=('ndcg_at_10', 'std'),
)
axes[0].errorbar(plot_df['n_classes'], plot_df['ndcg_mean'], yerr=plot_df['ndcg_std'], marker='o', linewidth=2)
axes[0].set_title('NDCG@10 vs confidence granularity')
axes[0].set_xlabel('n_classes')
axes[0].set_ylabel('NDCG@10')

for ax, n_classes in zip(axes[1:], [3, 6]):
    cms = [cm_store[(n_classes, seed)] for seed in SEEDS if (n_classes, seed) in cm_store]
    mean_cm = np.mean(cms, axis=0)
    labels = get_confidence_schema(n_classes)['class_names']
    sns.heatmap(mean_cm, annot=True, fmt='.1f', cmap='Blues', ax=ax, xticklabels=labels, yticklabels=labels)
    ax.set_title(f'{n_classes}-class confusion matrix')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Rule label')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'confidence_ablation_main_plots.png', dpi=200, bbox_inches='tight')
plt.show()

dist_rows = []
for (n_classes, seed), distribution in dist_store.items():
    for class_name, share in distribution.items():
        dist_rows.append({'n_classes': n_classes, 'seed': seed, 'class_name': class_name, 'share': share})
dist_df = pd.DataFrame(dist_rows)

plt.figure(figsize=(10, 5))
sns.barplot(data=dist_df, x='class_name', y='share', hue='n_classes', errorbar='sd')
plt.xticks(rotation=35, ha='right')
plt.title('Confidence class distribution by schema')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'confidence_ablation_distribution.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
skill_validation_rows = []
for seed in SEEDS:
    set_global_seed(seed)
    conf_agent = ConfidenceAgent(n_classes=6)
    conf_agent._config['seed'] = seed
    conf_agent.train(train_df)
    test_aug = attach_confidence_predictions(conf_agent, test_df)
    stats = validate_skill_deltas(test_aug, n_classes=6)
    stats['seed'] = seed
    skill_validation_rows.append(stats)

skill_validation_df = pd.DataFrame(skill_validation_rows)
skill_validation_df.to_csv(RESULTS_DIR / 'confidence_skill_delta_validation.csv', index=False)
display(skill_validation_df)

print(
    'Mean Spearman rho:',
    round(float(skill_validation_df['spearman_rho'].mean()), 4),
    '| Mean p-value:',
    round(float(skill_validation_df['p_value'].mean()), 4),
)